In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob

In [3]:
import pandas as pd
import os
import glob
import json

def parse_antibody_files_ighv3_21(directory_path):
    """
    Parse all CSV files and filter for IGHV3-21 sequences only
    """
    
    all_data = {}
    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))
    
    print(f"Found {len(csv_files)} CSV files to process...")
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        print(f"\nProcessing: {file_name}")
        
        try:
            # First, read without skipping rows to inspect the structure
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            # Find the header line (look for typical column names)
            header_line_index = 0
            for i, line in enumerate(lines):
                line_lower = line.lower()
                if ('sequence_id' in line_lower and 'cdr' in line_lower) or \
                   ('sequence_id_heavy' in line_lower) or \
                   ('v_call' in line_lower and 'j_call' in line_lower):
                    header_line_index = i
                    break
            
            print(f"  Detected header at line {header_line_index + 1}")
            
            # Read the file with the detected header
            df = pd.read_csv(file_path, skiprows=header_line_index, low_memory=False, skipinitialspace=True)
            
            # Clean column names
            df.columns = [col.strip() for col in df.columns]
            
            print(f"  Available columns: {list(df.columns)[:15]}...")  # Show first 15 columns
            
            # Define target columns and their possible aliases
            target_columns = {
                'sequence_id_heavy': ['sequence_id_heavy', 'sequence_id', 'seq_id_heavy', 'heavy_sequence_id'],
                'cdr1_heavy': ['cdr1_heavy', 'cdr1_h', 'heavy_cdr1', 'cdr1'],
                'cdr1_aa_heavy': ['cdr1_aa_heavy', 'cdr1_aa_h', 'heavy_cdr1_aa', 'cdr1_aa'],
                'cdr2_heavy': ['cdr2_heavy', 'cdr2_h', 'heavy_cdr2', 'cdr2'],
                'cdr2_aa_heavy': ['cdr2_aa_heavy', 'cdr2_aa_h', 'heavy_cdr2_aa', 'cdr2_aa'],
                'v_call_heavy': ['v_call_heavy', 'v_call', 'v_gene_heavy', 'heavy_v_call']  # Added V gene column
            }
            
            # Find actual column names in the file
            actual_columns = {}
            for target, possible_names in target_columns.items():
                found = None
                for possible in possible_names:
                    if possible in df.columns:
                        found = possible
                        break
                actual_columns[target] = found
            
            # Check which required columns we found
            missing_required = [target for target, actual in actual_columns.items() 
                              if actual is None and target.startswith('sequence_id')]
            
            if missing_required:
                print(f"  ❌ Missing required column: {missing_required}")
                print(f"  Found columns: {[f'{k}: {v}' for k, v in actual_columns.items() if v]}")
                continue
            
            # Check if we have V gene column for filtering
            if not actual_columns['v_call_heavy']:
                print(f"  ❌ No V gene column found for filtering IGHV3-21 sequences")
                print(f"  Looking for columns: {target_columns['v_call_heavy']}")
                continue
            
            # Process the data - FILTER FOR IGHV3-21 ONLY
            file_dict = {}
            sequences_processed = 0
            ighv3_21_count = 0
            
            seq_id_col = actual_columns['sequence_id_heavy']
            v_call_col = actual_columns['v_call_heavy']
            
            for index, row in df.iterrows():
                sequence_id = row[seq_id_col] if seq_id_col else None
                v_call = row[v_call_col] if v_call_col else None
                
                # Skip if no sequence ID
                if pd.isna(sequence_id) or sequence_id == '':
                    continue
                
                # FILTER: Only process sequences with IGHV3-21 gene
                if pd.isna(v_call) or v_call == '':
                    continue
                
                # Check if this sequence uses IGHV3-21 (allows for different naming conventions)
                v_call_str = str(v_call).upper()
                if 'IGHV3-21' in v_call_str or 'IGHV321' in v_call_str or 'V3-21' in v_call_str:
                    # Build sequence data dictionary
                    sequence_data = {}
                    for target, actual_col in actual_columns.items():
                        if actual_col and actual_col in df.columns:
                            value = row[actual_col]
                            sequence_data[target] = '' if pd.isna(value) else str(value).strip()
                        else:
                            sequence_data[target] = ''
                    
                    file_dict[str(sequence_id)] = sequence_data
                    sequences_processed += 1
                    ighv3_21_count += 1
                else:
                    # Count non-IGHV3-21 sequences for reporting
                    sequences_processed += 1
            
            all_data[file_name] = file_dict
            print(f"  ✅ Processed {sequences_processed} total sequences")
            print(f"  ✅ Found {ighv3_21_count} IGHV3-21 sequences")
            print(f"  Columns used: {[f'{k}→{v}' for k, v in actual_columns.items() if v]}")
            
        except Exception as e:
            print(f"  ❌ Error processing {file_name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    return all_data

def parse_antibody_files_ighv3_21_strict(directory_path):
    """
    Strict version that only includes exact IGHV3-21 matches
    """
    
    all_data = {}
    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))
    
    print(f"Found {len(csv_files)} CSV files to process...")
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        print(f"\nProcessing: {file_name}")
        
        try:
            # Find header line
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            header_line_index = 0
            for i, line in enumerate(lines):
                line_lower = line.lower()
                if ('sequence_id' in line_lower and 'cdr' in line_lower) or \
                   ('sequence_id_heavy' in line_lower) or \
                   ('v_call' in line_lower and 'j_call' in line_lower):
                    header_line_index = i
                    break
            
            print(f"  Detected header at line {header_line_index + 1}")
            
            # Read the file
            df = pd.read_csv(file_path, skiprows=header_line_index, low_memory=False, skipinitialspace=True)
            df.columns = [col.strip() for col in df.columns]
            
            # Find required columns
            target_columns = {
                'sequence_id_heavy': ['sequence_id_heavy', 'sequence_id', 'seq_id_heavy'],
                'cdr1_heavy': ['cdr1_heavy', 'cdr1_h', 'heavy_cdr1'],
                'cdr1_aa_heavy': ['cdr1_aa_heavy', 'cdr1_aa_h', 'heavy_cdr1_aa'],
                'cdr2_heavy': ['cdr2_heavy', 'cdr2_h', 'heavy_cdr2'],
                'cdr2_aa_heavy': ['cdr2_aa_heavy', 'cdr2_aa_h', 'heavy_cdr2_aa'],
                'v_call_heavy': ['v_call_heavy', 'v_call', 'v_gene_heavy']
            }
            
            actual_columns = {}
            for target, possible_names in target_columns.items():
                found = None
                for possible in possible_names:
                    if possible in df.columns:
                        found = possible
                        break
                actual_columns[target] = found
            
            # Check required columns
            if not actual_columns['sequence_id_heavy'] or not actual_columns['v_call_heavy']:
                print(f"  ❌ Missing required columns")
                continue
            
            # FILTER for IGHV3-21 sequences
            seq_id_col = actual_columns['sequence_id_heavy']
            v_call_col = actual_columns['v_call_heavy']
            
            # Convert V call to string and filter
            df['v_call_str'] = df[v_call_col].astype(str).str.upper()
            
            # Different patterns for IGHV3-21
            ighv3_21_patterns = [
                'IGHV3-21', 
                'IGHV321',
                'V3-21',
                '3-21'
            ]
            
            # Create filter mask
            mask = False
            for pattern in ighv3_21_patterns:
                mask = mask | df['v_call_str'].str.contains(pattern, na=False)
            
            # Apply filter
            ighv3_21_df = df[mask].copy()
            
            print(f"  Total sequences: {len(df)}")
            print(f"  IGHV3-21 sequences: {len(ighv3_21_df)}")
            
            # Process filtered sequences
            file_dict = {}
            for index, row in ighv3_21_df.iterrows():
                sequence_id = row[seq_id_col]
                
                if pd.isna(sequence_id) or sequence_id == '':
                    continue
                
                sequence_data = {}
                for target, actual_col in actual_columns.items():
                    if actual_col and actual_col in df.columns:
                        value = row[actual_col]
                        sequence_data[target] = '' if pd.isna(value) else str(value).strip()
                    else:
                        sequence_data[target] = ''
                
                file_dict[str(sequence_id)] = sequence_data
            
            all_data[file_name] = file_dict
            print(f"  ✅ Added {len(file_dict)} IGHV3-21 sequences")
            
        except Exception as e:
            print(f"  ❌ Error processing {file_name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    return all_data

def save_dictionary_to_json(data_dict, output_file):
    """Save the dictionary to a JSON file"""
    with open(output_file, 'w') as f:
        json.dump(data_dict, f, indent=2)
    print(f"\nDictionary saved to {output_file}")

def print_summary(data_dict):
    """Print summary of the parsed data"""
    print("\n" + "="*60)
    print("PARSING SUMMARY - IGHV3-21 SEQUENCES ONLY")
    print("="*60)
    
    total_files = len(data_dict)
    total_sequences = sum(len(sequences) for sequences in data_dict.values())
    
    print(f"Total files processed: {total_files}")
    print(f"Total IGHV3-21 sequences extracted: {total_sequences}")
    
    for file_name, sequences in data_dict.items():
        print(f"\n{file_name}: {len(sequences)} IGHV3-21 sequences")
        if sequences:
            first_seq_id = list(sequences.keys())[0]
            first_seq_data = sequences[first_seq_id]
            print(f"  Example - {first_seq_id}:")
            print(f"    V gene: {first_seq_data.get('v_call_heavy', 'N/A')}")
            for key, value in first_seq_data.items():
                if value and key not in ['v_call_heavy']:  # Don't show V gene again
                    print(f"    {key}: {value[:50]}{'...' if len(value) > 50 else ''}")

# Main execution
if __name__ == "__main__":
    # SET YOUR DIRECTORY PATH HERE
    directory_path = "Documents/hackaton/"  # ← CHANGE THIS TO YOUR ACTUAL PATH
    
    print("Parsing antibody sequencing files - IGHV3-21 sequences only...")
    
    # Use the strict version for better filtering
    antibody_data = parse_antibody_files_ighv3_21_strict(directory_path)
    
    # Print summary
    print_summary(antibody_data)
    
    # Save to JSON file
    if antibody_data:
        output_file = "antibody_ighv3-21_data.json"
        save_dictionary_to_json(antibody_data, output_file)
        
        # Additional statistics
        total_ighv3_21 = sum(len(seq_dict) for seq_dict in antibody_data.values())
        print(f"\n🎯 Successfully extracted {total_ighv3_21} IGHV3-21 sequences across {len(antibody_data)} files")
    else:
        print("\n❌ No IGHV3-21 sequences were found. Please check:")
        print("   - Your files contain V gene information")
        print("   - The V gene column is named correctly")
        print("   - There are sequences using IGHV3-21 gene")

Parsing antibody sequencing files - IGHV3-21 sequences only...
Found 236 CSV files to process...

Processing: GSM6504759_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 371
  IGHV3-21 sequences: 21
  ✅ Added 21 IGHV3-21 sequences

Processing: 1287146_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 19179
  IGHV3-21 sequences: 465
  ✅ Added 465 IGHV3-21 sequences

Processing: GSM6504701_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 8380
  IGHV3-21 sequences: 274
  ✅ Added 274 IGHV3-21 sequences

Processing: 1b_S2mod2_S1__1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2391
  IGHV3-21 sequences: 66
  ✅ Added 66 IGHV3-21 sequences

Processing: GSM6504689_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 3826
  IGHV3-21 sequences: 156
  ✅ Added 156 IGHV3-21 sequences

Processing: GSM6504736_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 1790
  IGHV3-21 sequences: 72
  ✅ Added 72 IGHV3-21 sequenc

  Total sequences: 5755
  IGHV3-21 sequences: 170
  ✅ Added 170 IGHV3-21 sequences

Processing: GSM6504695_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 843
  IGHV3-21 sequences: 26
  ✅ Added 26 IGHV3-21 sequences

Processing: GSM6504727_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2502
  IGHV3-21 sequences: 88
  ✅ Added 88 IGHV3-21 sequences

Processing: GSM6504749_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 4605
  IGHV3-21 sequences: 162
  ✅ Added 162 IGHV3-21 sequences

Processing: Human_colon_16S8157822_S53_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 292
  IGHV3-21 sequences: 10
  ✅ Added 10 IGHV3-21 sequences

Processing: GSM6504720_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2157
  IGHV3-21 sequences: 104
  ✅ Added 104 IGHV3-21 sequences

Processing: 3d_S6__1_Paired_All.csv
  Detected header at line 2
  Total sequences: 7464
  IGHV3-21 sequences: 263
  ✅ Added 263 IGHV3-21 sequences



  Total sequences: 1783
  IGHV3-21 sequences: 61
  ✅ Added 61 IGHV3-21 sequences

Processing: 1287144_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 18788
  IGHV3-21 sequences: 494
  ✅ Added 494 IGHV3-21 sequences

Processing: 1287150_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 16988
  IGHV3-21 sequences: 442
  ✅ Added 442 IGHV3-21 sequences

Processing: GSM7286920_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2109
  IGHV3-21 sequences: 88
  ✅ Added 88 IGHV3-21 sequences

Processing: 1287158_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 7327
  IGHV3-21 sequences: 187
  ✅ Added 187 IGHV3-21 sequences

Processing: GSM6504719_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2945
  IGHV3-21 sequences: 129
  ✅ Added 129 IGHV3-21 sequences

Processing: GSM6504770_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 5924
  IGHV3-21 sequences: 258
  ✅ Added 258 IGHV3-21 sequences

Processing: GSM

  Total sequences: 1218
  IGHV3-21 sequences: 28
  ✅ Added 28 IGHV3-21 sequences

Processing: GSM6504765_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 290
  IGHV3-21 sequences: 12
  ✅ Added 12 IGHV3-21 sequences

Processing: GSM6504685_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 1646
  IGHV3-21 sequences: 65
  ✅ Added 65 IGHV3-21 sequences

Processing: 1287159_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 18256
  IGHV3-21 sequences: 457
  ✅ Added 457 IGHV3-21 sequences

Processing: 2a_S6__1_Paired_All.csv
  Detected header at line 2
  Total sequences: 4748
  IGHV3-21 sequences: 164
  ✅ Added 164 IGHV3-21 sequences

Processing: GSM6504773_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 6699
  IGHV3-21 sequences: 324
  ✅ Added 324 IGHV3-21 sequences

Processing: 1e_S5__1_Paired_All.csv
  Detected header at line 2
  Total sequences: 8478
  IGHV3-21 sequences: 305
  ✅ Added 305 IGHV3-21 sequences

Processing: Pan_T7935

  Total sequences: 2498
  IGHV3-21 sequences: 96
  ✅ Added 96 IGHV3-21 sequences

Processing: GSM7286918_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 4233
  IGHV3-21 sequences: 140
  ✅ Added 140 IGHV3-21 sequences

Processing: GSM6504750_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 2435
  IGHV3-21 sequences: 88
  ✅ Added 88 IGHV3-21 sequences

Processing: Human_colon_16S8157815_S27_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 696
  IGHV3-21 sequences: 16
  ✅ Added 16 IGHV3-21 sequences

Processing: GSM7286917_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 4210
  IGHV3-21 sequences: 133
  ✅ Added 133 IGHV3-21 sequences

Processing: 1287154_1_Paired_All.csv
  Detected header at line 2
  Total sequences: 19839
  IGHV3-21 sequences: 551
  ✅ Added 551 IGHV3-21 sequences

Processing: 3c_S5__1_Paired_All.csv
  Detected header at line 2
  Total sequences: 7179
  IGHV3-21 sequences: 278
  ✅ Added 278 IGHV3-21 sequences

P


Dictionary saved to antibody_ighv3-21_data.json

🎯 Successfully extracted 37435 IGHV3-21 sequences across 236 files


In [4]:
def json_to_csv(json_file_path, csv_file_path=None):
    """
    Convert the antibody JSON data to a flattened CSV file
    """
    
    # Read the JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Create a list to store all flattened records
    flattened_data = []
    
    # Process each file and sequence
    for file_name, sequences in data.items():
        for sequence_id, sequence_data in sequences.items():
            # Create a base record with file and sequence info
            record = {
                'file_name': file_name,
                'sequence_id': sequence_id
            }
            
            # Add all sequence data fields
            record.update(sequence_data)
            
            flattened_data.append(record)
    
    # Convert to DataFrame
    df = pd.DataFrame(flattened_data)
    
    # Set output filename if not provided
    if csv_file_path is None:
        csv_file_path = json_file_path.replace('.json', '.csv')
    
    # Save to CSV
    df.to_csv(csv_file_path, index=False)
    
    print(f"✅ Converted JSON to CSV: {csv_file_path}")
    print(f"📊 Shape: {df.shape} (rows: {df.shape[0]}, columns: {df.shape[1]})")
    print(f"📋 Columns: {list(df.columns)}")
    
    return df

# Usage
if __name__ == "__main__":
    json_file = "antibody_ighv3-21_data.json"  # Your JSON file
    csv_file = "antibody_ighv3-21_data.csv"    # Output CSV file
    
    df = json_to_csv(json_file, csv_file)
    
    # Show a preview
    print("\n📄 First few rows:")
    print(df.head())

✅ Converted JSON to CSV: antibody_ighv3-21_data.csv
📊 Shape: (37435, 8) (rows: 37435, columns: 8)
📋 Columns: ['file_name', 'sequence_id', 'sequence_id_heavy', 'cdr1_heavy', 'cdr1_aa_heavy', 'cdr2_heavy', 'cdr2_aa_heavy', 'v_call_heavy']

📄 First few rows:
                     file_name                  sequence_id  \
0  GSM6504759_1_Paired_All.csv  TAGGCATTCAACGCTA-1_contig_1   
1  GSM6504759_1_Paired_All.csv  CACAGGCCAAGCGTAG-1_contig_2   
2  GSM6504759_1_Paired_All.csv  CTAATGGCAATGGACG-1_contig_1   
3  GSM6504759_1_Paired_All.csv  TTTGTCAGTTCGTGAT-1_contig_1   
4  GSM6504759_1_Paired_All.csv  ACGATGTTCCAAAGTC-1_contig_2   

             sequence_id_heavy                cdr1_heavy cdr1_aa_heavy  \
0  TAGGCATTCAACGCTA-1_contig_1  GGATTCACCTTCAGTAGCTATAGC      GFTFSSYS   
1  CACAGGCCAAGCGTAG-1_contig_2  GGCTTCACCTTCAGTACTTATAGT      GFTFSTYS   
2  CTAATGGCAATGGACG-1_contig_1  GGATTCACCTTCAGTAGCTATAGC      GFTFSSYS   
3  TTTGTCAGTTCGTGAT-1_contig_1  GGATTCACCTTCAGTAGCTATAGC      GFTFSSY